In [2]:
# %%
from llm.graph_transformer import LLMGraphTransformer
import util.helpers as helpers
import torch
import os
import pandas as pd
from langchain_community.graphs import Neo4jGraph

# %% [markdown]
# # Configure these parameters to select the appropriate graph and prompting strategy to use

# %%
from config import NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD,NEO4J_DATABASE, DIRECTORY
graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE, refresh_schema=False)

/mnt/danderson/home/pander14/tmp/ipykernel_1149304/1269888259.py:14: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE, refresh_schema=False)


In [4]:
import util.helpers as helpers
from langchain_community.vectorstores import Neo4jVector
from langchain_community.embeddings import OllamaEmbeddings, HuggingFaceEmbeddings
from graphdatascience import GraphDataScience

from prompts.construction_prompts.extraction_prompts import entity_resolution_prompt
from typing import List, Optional

In [5]:
edit_distance_query = helpers.load_query("edit_distance.cypher")
print(edit_distance_query)

# %%
word_edit_distance = 2
potential_duplicate_candidates = graph.query(edit_distance_query, params={'distance': word_edit_distance, 'min_length': 5})

# %%
from prompts.construction_prompts.extraction_prompts import entity_resolution_prompt
from llm.factory import get_client
from models.DisambiguateSchema import Disambiguate

extraction_llm = get_client(schema=Disambiguate)

# no retry decorator: APIClient.__call__ already retries once internally
def entity_resolution(entities: List[str]) -> Optional[List[str]]:
    res = extraction_llm(entity_resolution_prompt(sorted(entities)))
    return [
        el.entities
        for el in res.merge_entities
    ]

# %%
from tqdm import tqdm 

merged_entities = []

for el in tqdm(potential_duplicate_candidates, total=len(potential_duplicate_candidates), desc="Resolving entities"):
    merged_entities.extend(entity_resolution(el["combinedResult"]))

MATCH (e:`__Entity__`)
WHERE size(e.id) > $min_length
WITH e.wcc AS community, collect(e) AS nodes, count(*) AS count
WHERE count > 1
UNWIND nodes AS node
// Add text distance
WITH distinct
[n IN nodes WHERE apoc.text.distance(toLower(node.id), toLower(n.id)) < $distance | n.id] AS intermediate_results
WHERE size(intermediate_results) > 1
WITH collect(intermediate_results) AS results
// combine groups together if they share elements
UNWIND range(0, size(results)-1, 1) as index
WITH results, index, results[index] as result
WITH apoc.coll.sort(reduce(acc = result, index2 IN range(0, size(results)-1, 1) |
    CASE WHEN index <> index2 AND
	size(apoc.coll.intersection(acc, results[index2])) > 0
	THEN apoc.coll.union(acc, results[index2])
	ELSE acc
    END
)) as combinedResult
WITH distinct(combinedResult) as combinedResult
// extra filtering
WITH collect(combinedResult) as allCombinedResults
UNWIND range(0, size(allCombinedResults)-1, 1) as combinedResultIndex
WITH allCombinedResults[combi

Resolving entities: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 884/884 [05:32<00:00,  2.66it/s]


In [8]:
from collections.abc import Iterable


def consolidate_merge_groups(
    groups: Iterable[Iterable[str]],
) -> list[list[str]]:
    """
    Combine all transitively overlapping groups and remove duplicates.

    Examples:
        [A, B], [B, C] -> [A, B, C]
        [A, A]         -> removed
    """
    parent: dict[str, str] = {}
    rank: dict[str, int] = {}
    first_seen: dict[str, int] = {}
    sequence = 0

    def find(item: str) -> str:
        if parent[item] != item:
            parent[item] = find(parent[item])
        return parent[item]

    def add(item: str) -> None:
        nonlocal sequence

        if item not in parent:
            parent[item] = item
            rank[item] = 0
            first_seen[item] = sequence
            sequence += 1

    def union(left: str, right: str) -> None:
        left_root = find(left)
        right_root = find(right)

        if left_root == right_root:
            return

        if rank[left_root] < rank[right_root]:
            left_root, right_root = right_root, left_root

        parent[right_root] = left_root

        if rank[left_root] == rank[right_root]:
            rank[left_root] += 1

    for raw_group in groups:
        # Remove repeated IDs while preserving order.
        group = list(dict.fromkeys(raw_group))

        if not group:
            continue

        for entity_id in group:
            add(entity_id)

        for entity_id in group[1:]:
            union(group[0], entity_id)

    components: dict[str, list[str]] = {}

    for entity_id in parent:
        root = find(entity_id)
        components.setdefault(root, []).append(entity_id)

    cleaned = []

    for component in components.values():
        component.sort(key=first_seen.__getitem__)

        # A one-element component requires no merge.
        if len(component) > 1:
            cleaned.append(component)

    return cleaned


merged_entities = consolidate_merge_groups(merged_entities)

In [9]:
def validate_disjoint_groups(groups: list[list[str]]) -> None:
    owner: dict[str, int] = {}

    for group_index, group in enumerate(groups):
        if len(group) != len(set(group)):
            raise ValueError(
                f"Repeated ID inside group {group_index}: {group}"
            )

        for entity_id in group:
            if entity_id in owner:
                raise ValueError(
                    f"{entity_id!r} occurs in groups "
                    f"{owner[entity_id]} and {group_index}"
                )

            owner[entity_id] = group_index


validate_disjoint_groups(merged_entities)

In [10]:
merged_entities

[['Back Pain', 'Back_Pain', 'Backpain'],
 ['Roland Morris Disability Questionnaire',
  'Roland-Morris Disability Questionnaire'],
 ['African American', 'African-American'],
 ['African Americans', 'African-Americans'],
 ['Female', 'Females'],
 ['Older Age', 'Older_Age'],
 ['Employment Status', 'Employment_Status', 'Employmentstatus'],
 ['Smoking Status', 'Smoking_Status'],
 ['Current Smoker', 'Current_Smoker', 'Currentsmoker'],
 ['Current Smokers', 'Current_Smokers'],
 ['Pain Intensity', 'Pain_Intensity'],
 ['Pain Interference', 'Pain_Interference'],
 ['Psychological Distress', 'Psychologicaldistress'],
 ['Hispanic', 'Hispanics'],
 ['Caucasian', 'Caucasians'],
 ['Some College', 'Some_College'],
 ['Pe_Score', 'Peg Score', 'Peg_Score', 'Pegm_Score', 'Pegscore'],
 ['Sleep Disturbance', 'Sleep_Disturbance', 'sleepdisturbance'],
 ['Promis Depression', 'Promis_Depression', 'Promisdepression'],
 ['Promis Anxiety', 'Promis_Anxiety', 'Promisanxiety'],
 ['Fear Avoidance', 'Fear-Avoidance', 'Fear_

In [11]:
graph.query("""
UNWIND $data AS candidates
CALL {
  WITH candidates
  MATCH (e:__Entity__) WHERE e.id IN candidates
  RETURN collect(e) AS nodes
}
CALL apoc.refactor.mergeNodes(nodes, {properties: {
    `.*`: 'discard'
}})
YIELD node
RETURN count(*)
""", params={"data": merged_entities})

[{'count(*)': 905}]

In [14]:
# %%
# project graph
gds = GraphDataScience(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD), database=NEO4J_DATABASE

)

# Drop GDS projections if they exist
try:
    gds.graph.drop("entities")
except:
    pass
try:
    gds.graph.drop("communities")
except:
    pass

# %%
G, result = gds.graph.project(
    "entities",                   #  Graph name
    "__Entity__",                 #  Node projection
    "*",                          #  Relationship projection
    nodeProperties=["embedding"]  #  Configuration parameters
)

# %%
gds.graph.drop("entities")

# %%
G.drop()

# %% [markdown]
# # Communities

# %%
gds.graph.drop('communities')

# %%
G, result = gds.graph.project(
    "communities",  #  Graph name
    "__Entity__",  #  Node projection
    {
        "_ALL_": {
            "type": "*",
            "orientation": "UNDIRECTED",
            "properties": {"weight": {"property": "*", "aggregation": "COUNT"}},
        }
    },
)

# %%
wcc = gds.wcc.stats(G)
print(f"Component count: {wcc['componentCount']}")
print(f"Component distribution: {wcc['componentDistribution']}")

# %%
gds.leiden.write(
    G,
    writeProperty="communities",
    includeIntermediateCommunities=True,
    relationshipWeightProperty="weight",
    randomSeed=42,
    concurrency=1,
    theta=0
)

# %%
graph.query("CREATE CONSTRAINT IF NOT EXISTS FOR (c:__Community__) REQUIRE c.id IS UNIQUE;")

# %%
# creates community nodes and adds edges from each node to the community it belongs too. 
graph.query("""
MATCH (e:`__Entity__`)
UNWIND range(0, size(e.communities) - 1 , 1) AS index
CALL {
  WITH e, index
  WITH e, index
  WHERE index = 0
  MERGE (c:`__Community__` {id: toString(index) + '-' + toString(e.communities[index])})
  ON CREATE SET c.level = index
  MERGE (e)-[:IN_COMMUNITY]->(c)
  RETURN count(*) AS count_0
}
CALL {
  WITH e, index
  WITH e, index
  WHERE index > 0
  MERGE (current:`__Community__` {id: toString(index) + '-' + toString(e.communities[index])})
  ON CREATE SET current.level = index
  MERGE (previous:`__Community__` {id: toString(index - 1) + '-' + toString(e.communities[index - 1])})
  ON CREATE SET previous.level = index - 1
  MERGE (previous)-[:IN_COMMUNITY]->(current)
  RETURN count(*) AS count_1
}
RETURN count(*)
""")

# %%
graph.query("MATCH (n:`__Community__`) return count(n)")

# %%
# community structure
# finds the size of all communities with more than two entities
community_size_df = graph.query(
    """
    MATCH (c:__Community__)<-[:IN_COMMUNITY*]-(e:__Entity__)
    WITH c, count(distinct e) AS entities
    WHERE entities > 1
    RETURN split(c.id, '-')[0] AS level, entities
    """
)
community_size_df = pd.DataFrame(community_size_df)
helpers.community_analysis(community_size_df)

# %% [markdown]
# # Summarizing Communities

# %%
community_info = graph.query("""
MATCH (c:`__Community__`)<-[:IN_COMMUNITY*]-(e:__Entity__)
WITH c, collect(e) AS nodes
WHERE size(nodes) > 1
CALL apoc.path.subgraphAll(nodes[0], {
	whitelistNodes:nodes
})
YIELD relationships
RETURN c.id AS communityId, c.level as level,
       [r in relationships | {
                             start_id: startNode(r).id,
                             start_desc: startNode(r).description, 
                             rel_desc: r.description, 
                             rel_type: type(r),
                             end_id: endNode(r).id,
                             end_desc: endNode(r).description, 
                             degree: apoc.node.degree(startNode(r)) + apoc.node.degree(endNode(r))
                             }] AS triplets
""")

# %%
# first pass: sort the triplets, separate into two groups: leaves and nonleaves
context_window_limit = 8000

# %%
from context_construction.build_context import split_and_sort, summarize_leaves, normalize_nonleaves, summarize_nonleaves

raw_leaves, raw_nonleaves = split_and_sort(community_info) 
leaves_with_reports, leaves_reports_map = summarize_leaves(raw_leaves, context_window_limit)

# %%
nonleaves_normalized = normalize_nonleaves(raw_nonleaves, leaves_reports_map)
nonleaves_with_reports = summarize_nonleaves(nonleaves_normalized, context_window_limit)

# %%
from context_construction.build_context import normalize_summarized_community

leaves = list(
    map(normalize_summarized_community, leaves_with_reports)
)

nonleaves = list(
    map(normalize_summarized_community, nonleaves_with_reports)
)

# %%
# Store summaries
graph.query("""
UNWIND $data AS row
MERGE (c:__Community__ {id:row.community})
SET c.title=row.title, c.summary = row.summary, c.impact_severity_rating=row.impact_severity_rating, c.rating_explanation=row.rating_explanation, c.detailed_findings=row.detailed_findings
""", params={"data": leaves + nonleaves})

# %% [markdown]
# # Weighting the Graph

# %%
# community rank set to be the number of documents referenced by that community
graph.query("""
MATCH (c:__Community__)<-[:IN_COMMUNITY*]-(:__Entity__)<-[:MENTIONS]-(d:Document)
WITH c, count(distinct d) AS rank
SET c.community_rank = rank;
""")


Component count: 2172
Component distribution: {'min': 1, 'p5': 1, 'max': 13049, 'p999': 21, 'p99': 7, 'p1': 1, 'p10': 1, 'p90': 3, 'p50': 1, 'p25': 1, 'p75': 2, 'p95': 4, 'mean': 7.64963167587477}


Building Leaf Contexts: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2329/2329 [00:01<00:00, 1971.99it/s]


APIClient.map:   0%|          | 0/2329 [00:00<?, ?it/s]

Normalizing Non Leaves


Nonleaf Normalization Progress: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3666/3666 [01:24<00:00, 43.35it/s]


...

Nonleaf Context Building Progress: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3666/3666 [00:02<00:00, 1489.87it/s]


Summarizing Nonleaves


APIClient.map:   0%|          | 0/3666 [00:00<?, ?it/s]

Error in APIClient call, retrying once: Request timed out.
Error in APIClient.map() item 334, skipping: Request timed out.
Error in APIClient call, retrying once: Request timed out.
Error in APIClient call, retrying once: Request timed out.
Error in APIClient call, retrying once: Request timed out.
Error in APIClient.map() item 1327, skipping: Request timed out.
Finished Summarizing Non Leaves


[]